[Source](https://docs.nvidia.com/bionemo-framework/1.10/notebooks/MolMIM_GenerativeAI_local_inference_with_examples.html)

In [1]:
import os
import itertools
import warnings
import logging
import random

import torch
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
import numpy as np

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.molmim.infer import MolMIMInference

# random.seed(42)
# np.random.seed(42)
# torch.manual_seed(42)

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
[NeMo W 2025-08-08 08:57:49 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:257: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
      def forward(
    
[NeMo W 2025-08-08 08:57:49 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:268: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
      def backward(ctx, grad_output):
    
[NeMo W 2025-08-08 08:57:49 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:328: FutureWarni

[NeMo I 2025-08-08 08:57:54 megatron_hiddens:110] Registered hidden transform sampled_var_cond_gaussian at bionemo.model.core.hiddens_support.SampledVarGaussianHiddenTransform
[NeMo I 2025-08-08 08:57:54 megatron_hiddens:110] Registered hidden transform interp_var_cond_gaussian at bionemo.model.core.hiddens_support.InterpVarGaussianHiddenTransform


In [2]:
bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

In [3]:
torch.cuda.is_available()

True

In [15]:
# !python download_artifacts.py --model_dir ${BIONEMO_HOME}/models --models molmim_70m_24_3

In [4]:
# %%capture --no-display --no-stderr cell_output

# Load pre-trained model checkpoints
checkpoint_path = f"{bionemo_home}/models/molecule/molmim/molmim_70m_24_3.nemo"
# checkpoint_path = f"{bionemo_home}/data/models/MolMIM-MolMIM-small--val_molecular_accuracy=0.91-val_loss=-0.31-step=25245-consumed_samples=25850880.0-last.nemo"


# Load starting config for MolMIM inference
cfg = load_model_config(
    config_name="molmim_infer.yaml",
    config_path=f"{bionemo_home}/examples/tests/conf/"
)

# Point YAML configuration file to the location of the desired checkpoints
cfg.model.downstream_task.restore_from_path = checkpoint_path
#cfg.model.encoder.hidden_steps = 2

print(os.path.isfile(checkpoint_path))

# Create model object based on desired configuration
model = MolMIMInference(cfg, interactive=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


True
Interactive mode selected, using strategy='auto'


[W808 08:58:03.562113092 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
25-08-08 08:58:03 - PID:23774 - rank:(0, 0, 0, 0) - microbatches.py:39 - INFO - setting number of micro-batches to constant 1


In [21]:
expert_smiles_path = "/workspace/bionemo/data/train_substrate.csv"
expert_smiles_filename = expert_smiles_path.split("/")[-1]
expert_smiles = pd.read_csv(expert_smiles_path)['canon_smiles'].tolist()

# expert_smiles = [smiles for smiles in expert_smiles if len(smiles) <= 128]

len(expert_smiles)

82

In [18]:
def chem_sample(
        reference_smiles: list[str],
        num_samples: int = 100,
        sampling_method: str = "beam-search-perturbate",
        **sampler_kwargs
) -> list:
    # PART 1: SAMPLING
    # 1A: Set Sampling Arguments
    # default_sampler_kwargs = {"beam_size": 3, "keep_only_best_tokens": True, "return_scores": False}
    # sampler_kwargs = {**default_sampler_kwargs, **sampler_kwargs}  # Override defaults with user-provided kwargs

    # 1B: Execute sampling

    try:

        population_samples = model.sample(
            seqs=reference_smiles,
            num_samples=num_samples,
            sampling_method=sampling_method,
            **sampler_kwargs
        )

        return population_samples

    except Exception as e:
        logging.error(f"Error during sampling: {e}")
        return []  # Return an empty list in case of error

### Sampling methods:
- "greedy-perturbate": Sample the best sequence for each of our perturbed hiddens, using greedy-search (per token)
to find the best result.
- "topkp-perturbate": Sample the best sequence for each of our perturbed hiddens, using `topkp-sampling` to
find the best result.
- "beam-search-perturbate": Sample the best sequence for each of our perturbed hiddens,
using beam-search to find the best result.
 - "beam-search-single-sample": Sample the top num_samples sequences using beam-search (rather than single best) given our
single perturbed hidden.
- "beam-search-perturbate-sample": Sample the top beam_size (default 5) sequences using beam-search
for each of our `num_samples` purturbed hiddens. This will return a `beam_size * num_samples` set of results.

### Params
```
{
            # seqs - a list of Sequence strings to perturbate num_samples times each (could be SMILE, protein AA, etc)
            "greedy-perturbate": {"scaled_radius": 1, "seqs": [], "hiddens": None, "enc_masks": None},
            # top-k limits maximum number of token candidtaes, top-p can further reduce to accumulate top-p probability mass
            "topkp-perturbate": {
                "scaled_radius": 1,
                "seqs": [],
                "top_k": 0,
                "top_p": 0.9,
                "temperature": 1.0,
                "hiddens": None,
                "enc_masks": None,
            },
            # Beam search perturbate works the same way as `greedy-perturbate` but is more likely to find the global optima
            #  of argmax_sequence P(sequence|hiddens). The best result is returned greedily, but the search is less biased
            #  by the best tokens at prior states.
            "beam-search-perturbate": {
                "scaled_radius": 1,
                "seqs": [],
                "beam_size": 5,  # This strategy uses the same perturb hiddens then decode process, so beam_size is independent now
                "keep_only_best_tokens": True,  # Now we only return the best result, greedily, from beam-search.
                "beam_alpha": 0,
                "hiddens": None,
                "enc_masks": None,
            },
            # beam search single sample, "beam_size" is number of the best sequences at each decode iteration to be left per target
            # and "beam_alpha" is the parameter of length penalty applied to predicted sequences.
            # NOTE with this method we only draw one gaussian sample of the hidden state, and then use beam search to return the
            #  top num_samples results. You probably want to use `beam-search-perturbate`.
            "beam-search-single-sample": {
                "scaled_radius": 1,
                "seqs": [],
                "beam_size": 1,  # will be set to num_samples in code. If left as 1 this is basically greedy-search
                "beam_alpha": 0,
                "keep_only_best_tokens": False,  # this strategy returns all of the beam search internal top_k results
                "hiddens": None,
                "enc_masks": None,
            },
            # beam search perturbate sample, "beam_size" is number of the best sequences at each decode iteration to be left per target
            # and "beam_alpha" is the parameter of length penalty applied to predicted sequences.
            # NOTE with this method we only draw one gaussian sample of the hidden state, and then use beam search to return the
            #  top num_samples results. You probably want to use `beam-search-perturbate`.
            "beam-search-perturbate-sample": {
                "scaled_radius": 1,
                "seqs": [],
                "beam_size": 5,  # this is the number of top samples to return for each num_sample hidden.
                "beam_alpha": 0,
                "keep_only_best_tokens": False,  # this strategy returns all of the beam search internal top_k results
                "hiddens": None,
                "enc_masks": None,
            },
        }
```


In [ ]:
# model.default_sampling_kwargs # run to see the dictionary above

## Define sampling arguments

In [7]:
param_grid = {
    # "greedy-perturbate": {
    #     "scaled_radius": [0.5, 2],
    # },
    # "topkp-perturbate": {
    #     "scaled_radius": [1, 2],
    #     "top_k": [5, 50],
    #     "top_p": [0.8, 1.0],
    #     "temperature": [0.7, 1.0],
    # },
    # "beam-search-perturbate": {
    #     "scaled_radius": [1, 2],
    #     "beam_size": [3, 5],
    #     "beam_alpha": [0, 0.5],
    # },
        "beam-search-perturbate": {
        "scaled_radius": [1],
        "beam_size": [3],
        "beam_alpha": [0.5],
    },
    # "beam-search-single-sample": {
    #     "scaled_radius": [1, 2],
    #     "beam_size": [1, 5],
    #     "beam_alpha": [0, 0.5],
    # },
    # "beam-search-perturbate-sample": {
    #     "scaled_radius": [1, 2],
    #     "beam_size": [5, 10],
    #     "beam_alpha": [0, 0.5],
    # }
}


In [8]:
def format_kwargs(kwargs: dict) -> str:
    return "_".join([f"{k}_{v}" for k, v in kwargs.items()])

In [9]:
def extract_param_set(method: str, index: int = 0) -> dict:
    """
    Extracts a specific parameter set for the given sampling method from the parameter grid.

    Args:
        method (str): The name of the sampling method.
        index (int): The index of the parameter set to extract (e.g., 0 for the first, 1 for the second).

    Returns:
        dict: A dictionary containing the selected parameter set.
    """
    if method not in param_grid:
        raise ValueError(f"Method '{method}' not found in parameter grid.")

    params = {}
    for key, values in param_grid[method].items():
        if index >= len(values):
            raise IndexError(f"Index {index} is out of range for parameter '{key}' in method '{method}'.")
        params[key] = values[index]
    return params


## Create chunks:
### 1. Generate chunks of specified size

In [ ]:
# # # Specify the chunk sizes manually for the first few chunks
# chunk_sizes = [14, 10]  # 18 molecules in the first chunk, 6 in the second (for example)
#
# # Calculate the total number of molecules
# total_molecules = len(expert_smiles)
#
# # Calculate how many molecules are left after the initial chunks
# molecules_left = total_molecules - sum(chunk_sizes)
#
# # If there are molecules left, create a final chunk with all remaining molecules
# if molecules_left > 0:
#     chunk_sizes.append(molecules_left)
#
# # Divide the expert_smiles into the specified chunks
# chunks = []
# start = 0
# for size in chunk_sizes:
#     end = start + size
#     chunks.append(expert_smiles[start:end])
#     start = end
# print([len(chunk) for chunk in chunks])

### 2. Generate N equal chunks

In [23]:
n = 82
chunks = np.array_split(expert_smiles, n)

len(chunks)

82

In [24]:
fine_tuned_str = "molmim_vanilla"  # Add any additional fine-tuning string if needed
num_samples: int = 100


for i in range(10):

    for method in param_grid.keys():
        num_param_sets = len(next(iter(param_grid[method].values())))

        for index in range(num_param_sets):
            param_dict = extract_param_set(method, index=index)
            print(f"Running {method} with parameter set {index}: {param_dict}")

            gen_smis_lst = []
            for chunk in tqdm(chunks, total=len(chunks)):
               # Call chem_sample with the current combination of parameters
               gen_smis = chem_sample(chunk, num_samples=num_samples, **param_dict)
               gen_smis_lst.extend(gen_smis)

            # Flatten the list of generated SMILES
            flattened = list(itertools.chain.from_iterable(gen_smis_lst))

            # Check if any valid molecules were generated
            if len(flattened) == 0:
               print("No molecules generated.")
               continue

            # Format the kwargs and generate the file name
            filename = f"data/outputs/bionemo_molmim_{expert_smiles_filename.split('.')[0]}_{fine_tuned_str}_{i}_num_samples_{num_samples}_sampling_method_{method}_{format_kwargs(param_dict)}.csv"

            # Save the generated molecules to a CSV file
            pd.DataFrame(
                    data={"SMILES": flattened}
                ).to_csv(filename, index=False)

            # Optionally print or log the saved file name
            print(f"Saved to: {filename}")

Running beam-search-perturbate with parameter set 0: {'scaled_radius': 1, 'beam_size': 3, 'beam_alpha': 0.5}


  6%|▌         | 5/82 [00:23<05:56,  4.63s/it]ERROR:root:Error during sampling: One or more sequence exceeds max length(128).
ERROR:root:Error during sampling: One or more sequence exceeds max length(128).
 88%|████████▊ | 72/82 [04:59<00:43,  4.39s/it]ERROR:root:Error during sampling: One or more sequence exceeds max length(128).
ERROR:root:Error during sampling: One or more sequence exceeds max length(128).
100%|██████████| 82/82 [05:30<00:00,  4.04s/it]

Saved to: data/outputs/bionemo_molmim_train_substrate_molmim_vanilla_9_num_samples_100_sampling_method_beam-search-perturbate_scaled_radius_1_beam_size_3_beam_alpha_0.5.csv
